<a href="https://colab.research.google.com/github/Nikhilsankhyan0/Agrisheild-AI/blob/main/model_artifacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import joblib
import os

In [6]:
x_train = pd.read_csv(
    "/content/drive/MyDrive/X_train_encoded.csv"
)

x_val = pd.read_csv(
    "/content/drive/MyDrive/X_val_encoded.csv"
)

x_test = pd.read_csv(
    "/content/drive/MyDrive/X_test_encoded.csv"
)

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(14000, 259)
(3000, 259)
(3000, 259)


target files

In [7]:
y_train_reg = pd.read_csv(
    "/content/drive/MyDrive/y_train_reg.csv"
).squeeze()

y_val_reg = pd.read_csv(
    "/content/drive/MyDrive/y_val_reg.csv"
).squeeze()

y_test_reg = pd.read_csv(
    "/content/drive/MyDrive/y_test_reg.csv"
).squeeze()


y_train_cls = pd.read_csv(
    "/content/drive/MyDrive/y_train_cls.csv"
).squeeze()

y_val_cls = pd.read_csv(
    "/content/drive/MyDrive/y_val_cls.csv"
).squeeze()

y_test_cls = pd.read_csv(
    "/content/drive/MyDrive/y_test_cls.csv"
).squeeze()

check saved models

In [8]:
import os

files = os.listdir("/content/drive/MyDrive")

for file in files:
    if file.endswith(".pkl"):
        print(file)

tfidf.pkl
tfidf_model.pkl
semantic_model.pkl
final_Risk_score_model.pkl
final_xgb_classifier.pkl
classification_label_encoder.pkl


load the models

In [9]:
import joblib

# Regression model
risk_score_model = joblib.load(
    "/content/drive/MyDrive/final_Risk_score_model.pkl"
)

# Classification model
xgb_classifier = joblib.load(
    "/content/drive/MyDrive/final_xgb_classifier.pkl"
)

# Classification label encoder
label_encoder = joblib.load(
    "/content/drive/MyDrive/classification_label_encoder.pkl"
)

print("All final artifacts loaded successfully.")

All final artifacts loaded successfully.


In [10]:
print("Risk Score Model:")
print(risk_score_model)

print("\nClassification Model:")
print(xgb_classifier)

print("\nClasses:")
print(label_encoder.classes_)

Risk Score Model:
ElasticNet(alpha=0.01, max_iter=10000, random_state=42)

Classification Model:
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

Classes:
['Approved' 'Approved with Conditions' 'Rejected']


test on test data

In [11]:
# ============================================================
# FINAL MODEL VERIFICATION
# ============================================================

# Regression prediction
risk_pred = risk_score_model.predict(x_test)

# Classification prediction
cls_pred_encoded = xgb_classifier.predict(x_test)

# Convert classification numbers back to labels
cls_pred = label_encoder.inverse_transform(
    cls_pred_encoded.astype(int)
)

print("===== REGRESSION =====")
print("First 5 Risk Score predictions:")
print(risk_pred[:5])

print("\n===== CLASSIFICATION =====")
print("First 5 encoded predictions:")
print(cls_pred_encoded[:5])

print("\nFirst 5 actual labels:")
print(y_test_cls[:5].values)

print("\nFirst 5 predicted labels:")
print(cls_pred[:5])

===== REGRESSION =====
First 5 Risk Score predictions:
[208065.06328439 421820.18632405 110116.05086836 675466.7422766
 155761.84301387]

===== CLASSIFICATION =====
First 5 encoded predictions:
[0 2 2 0 2]

First 5 actual labels:
['Approved with Conditions' 'Rejected' 'Rejected'
 'Approved with Conditions' 'Rejected']

First 5 predicted labels:
['Approved' 'Rejected' 'Rejected' 'Approved' 'Rejected']


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

In [13]:
risk_pred = risk_score_model.predict(x_test_scaled)

print(risk_pred[:5])

[48.02789997 26.24944274 20.37635447 46.26426573 29.29044033]


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test_reg, risk_pred)
rmse = np.sqrt(mean_squared_error(y_test_reg, risk_pred))
r2 = r2_score(y_test_reg, risk_pred)

print("===== SAVED REGRESSION MODEL =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

===== SAVED REGRESSION MODEL =====
MAE  : 4.1369
RMSE : 5.2320
R²   : 0.8221


In [15]:
xgb_classifier.predict(x_test)

array([0, 2, 2, ..., 1, 1, 0])

In [17]:
label_encoder.inverse_transform(cls_pred_encoded)

array(['Approved', 'Rejected', 'Rejected', ...,
       'Approved with Conditions', 'Approved with Conditions', 'Approved'],
      dtype=object)